# Phase 3 RQ1 v2 — combined analysis

This analysis stage runs on CPU; no model training or GPU is needed. It analyzes the same ten-language Phase 4 roster: `en`, `de`, `es`, `fr`, `sw`, `ar`, `ur`, `bg`, `ru`, and `hi`. No GitHub access or Internet connection is required. Attach the supplied outer ZIP containing the local analysis code, `phase3_rq1_v2_inputs.zip`, and the 15 per-order result ZIPs. If Kaggle extracts those archives automatically, the notebook rebuilds the analysis ZIPs from their extracted contents. The workflow also accepts one complete seed ZIP instead of five per-order ZIPs, but every seed must have all five orders represented exactly once.

The workflow checks that the seeds, five orders per seed, 15 result records, and data-manifest hashes match the frozen protocol before calculating pooled and per-order correlations, 10,000 language-label permutations, Holm correction, and a random-score control. It exports `phase3_rq1_v2_analysis.zip` under `/kaggle/working`. Review the per-seed metric summaries in the provenance and the per-order correlation table alongside the pooled corrected results; this pilot does not automatically select a prior.

In [3]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from zipfile import ZipFile

INPUT_ROOT = Path("/kaggle/input")
CODE_ARCHIVE_NAME = "phase3_rq1_v2_analysis_code.zip"
code_archives = list(INPUT_ROOT.rglob(CODE_ARCHIVE_NAME))
if not code_archives:
    embedded_code = []
    for outer_zip in INPUT_ROOT.rglob("*.zip"):
        with ZipFile(outer_zip) as archive:
            members = [name for name in archive.namelist() if Path(name).name == CODE_ARCHIVE_NAME]
            for member in members:
                extracted_code = Path("/kaggle/working") / CODE_ARCHIVE_NAME
                with archive.open(member) as source, extracted_code.open("wb") as target:
                    shutil.copyfileobj(source, target)
                embedded_code.append(extracted_code)
    code_archives = embedded_code
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if len(code_archives) == 1:
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    with ZipFile(code_archives[0]) as archive:
        archive.extractall(REPO_ROOT)
elif not code_archives:
    workflow_files = [
        path for path in INPUT_ROOT.rglob("rq1_v2_workflow.py")
        if path.parent.name == "phase3" and (path.parents[1] / "phase2").is_dir()
    ]
    if len(workflow_files) == 1:
        REPO_ROOT = workflow_files[0].parents[1]
    else:
        visible_files = [str(path) for path in INPUT_ROOT.rglob("*") if path.is_file()]
        raise FileNotFoundError(
            f"Expected {CODE_ARCHIVE_NAME} or an extracted project source tree. "
            f"Found code archives: {code_archives}. Files under /kaggle/input: {visible_files}"
        )
else:
    raise ValueError(f"Expected one {CODE_ARCHIVE_NAME}; found: {code_archives}")

if not (REPO_ROOT / "phase3" / "rq1_v2_workflow.py").is_file():
    raise FileNotFoundError("Bundled analysis source is missing phase3/rq1_v2_workflow.py")
sys.path.insert(0, str(REPO_ROOT))
INPUT_ZIP_NAME = "phase3_rq1_v2_inputs.zip"
RESULT_PREFIX = "phase3_rq1_v2_b0_seed_"
inputs = list(INPUT_ROOT.rglob(INPUT_ZIP_NAME))
if not inputs:
    input_summaries = list(INPUT_ROOT.rglob("rq1_v2_input_summary.json"))
    if len(input_summaries) == 1:
        input_root = input_summaries[0].parent
        input_bundle = Path("/kaggle/working") / INPUT_ZIP_NAME
        with ZipFile(input_bundle, "w") as archive:
            for path in sorted(input_root.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(input_root).as_posix())
        inputs = [input_bundle]
        print(f"Repacked the extracted input dataset: {input_bundle}")
    elif len(input_summaries) > 1:
        raise ValueError(f"Expected one extracted input dataset; found: {input_summaries}")

if not inputs:
    embedded_inputs = []
    for outer_zip in INPUT_ROOT.rglob("*.zip"):
        with ZipFile(outer_zip) as archive:
            members = [name for name in archive.namelist() if Path(name).name == INPUT_ZIP_NAME]
            for member in members:
                extracted_input = Path("/kaggle/working") / INPUT_ZIP_NAME
                with archive.open(member) as source, extracted_input.open("wb") as target:
                    shutil.copyfileobj(source, target)
                embedded_inputs.append(extracted_input)
    inputs = embedded_inputs

if len(inputs) != 1:
    visible_files = [str(path) for path in INPUT_ROOT.rglob("*") if path.is_file()]
    raise FileNotFoundError(
        "Attach exactly one dataset containing phase3_rq1_v2_inputs.zip, "
        "its extracted files, or an outer ZIP that contains it. "
        f"Found input bundles: {inputs}. Files under /kaggle/input: {visible_files}"
    )

result_zips = sorted(INPUT_ROOT.rglob(f"{RESULT_PREFIX}*.zip"))
result_names = {path.name for path in result_zips}
extracted_results = Path("/kaggle/working/phase3_rq1_v2_result_zips")
extracted_results.mkdir(parents=True, exist_ok=True)
for summary_path in INPUT_ROOT.rglob("rq1_v2_b0_order_summary.json"):
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    seed = summary.get("seed")
    order_id = summary.get("order_id")
    if seed not in (42, 43, 44) or order_id not in {f"order-{index}" for index in range(1, 6)}:
        raise ValueError(f"Invalid seed/order in extracted result summary: {summary_path}")
    result_name = f"{RESULT_PREFIX}{seed}_{order_id}.zip"
    if result_name not in result_names:
        result_path = extracted_results / result_name
        with ZipFile(result_path, "w") as archive:
            for path in sorted(summary_path.parent.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(summary_path.parent).as_posix())
        result_zips.append(result_path)
        result_names.add(result_name)
for outer_zip in INPUT_ROOT.rglob("*.zip"):
    with ZipFile(outer_zip) as archive:
        for member in archive.namelist():
            name = Path(member).name
            if name.startswith(RESULT_PREFIX) and name.endswith(".zip") and name not in result_names:
                result_path = extracted_results / name
                with archive.open(member) as source, result_path.open("wb") as target:
                    shutil.copyfileobj(source, target)
                result_zips.append(result_path)
                result_names.add(name)
result_zips = sorted(result_zips)

if len(result_zips) < 3 or len(result_zips) > 15:
    visible_files = [str(path) for path in INPUT_ROOT.rglob("*") if path.is_file()]
    raise FileNotFoundError(
        "Attach either three complete seed ZIPs or 15 per-order result ZIPs, "
        f"individually or inside one outer ZIP. Found {len(result_zips)}: {result_zips}. "
        f"Files under /kaggle/input: {visible_files}"
    )

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_analysis")
command = [
    sys.executable, "-m", "phase3.rq1_v2_workflow", "analyze",
    "--inputs-zip", str(inputs[0]),
    "--output-dir", str(OUTPUT_DIR),
    "--random-repetitions", "10000",
]
for result_zip in result_zips:
    command.extend(["--result-zip", str(result_zip)])
subprocess.run(command, cwd=REPO_ROOT, check=True)

ANALYSIS_ZIP = OUTPUT_DIR / "phase3_rq1_v2_analysis.zip"
if not ANALYSIS_ZIP.is_file():
    raise FileNotFoundError(f"Analysis bundle was not created: {ANALYSIS_ZIP}")
print(f"Download the combined analysis from Kaggle output files: {ANALYSIS_ZIP}")

Repacked the extracted input dataset: /kaggle/working/phase3_rq1_v2_inputs.zip
Analyzed 15 B0 runs and 90 directed language pairs.
Results: /kaggle/working/phase3_rq1_v2_analysis/analysis
{
  "analysis": "phase3_rq1_prior_vs_forgetting",
  "correlations": [
    {
      "directed_language_pairs": 90,
      "forgetting_observations": 675,
      "order_count": 5,
      "p_holm": 1.0,
      "p_one_sided": 0.6146385361463854,
      "permutation_count": 10000,
      "prior": "script_mismatch",
      "spearman_rho": -0.03578498557039991,
      "test_method": "monte_carlo_language_label_permutation"
    },
    {
      "directed_language_pairs": 90,
      "forgetting_observations": 675,
      "order_count": 5,
      "p_holm": 1.0,
      "p_one_sided": 0.3104689531046895,
      "permutation_count": 10000,
      "prior": "family_mismatch",
      "spearman_rho": 0.12350122764163006,
      "test_method": "monte_carlo_language_label_permutation"
    },
    {
      "directed_language_pairs": 90,
    